# Phụ lục A — Tái lập Thực nghiệm và Cấu hình Google Colab CPU-only

## A.1. Biên dịch lại báo cáo Jupyter Book (độc lập, 0 cần tải 22,72 GiB corpus thô)

Mọi bảng thống kê và biểu đồ trong cuốn sách này đều được tính toán và vẽ trực tiếp từ tệp tổng hợp `report_summary.json` cùng các đoạn mã gọi thư viện `parser` thật ngay tại thời điểm biên dịch (`execute_notebooks: force` dưới cờ nghiêm ngặt `-W`):

```bash
# Cách 1: Biên dịch trực tiếp từ kho lưu trữ Jupyter Book
python -m pip install -r requirements.txt
jupyter-book build . --all -W

# Cách 2: Sinh lại toàn bộ mã nguồn Book từ kho mã nguồn chính (nhánh main)
git clone https://github.com/NguyenQuocDat06-AI/arxiv-harvester.git
cd arxiv-harvester
python -m pip install -e ".[report]"
python tools/build_team_25000_book.py --workspace build/m2_workspace --output build/m2_book_staging --build
```

## A.2. Tái lập thu thập dữ liệu Milestone 1 (CLI & Google Colab CPU-only)

Trên máy cục bộ hoặc runtime **Google Colab CPU-only** (`colab_crawler.ipynb`, `kaggle_crawler.ipynb`, `tools/m1_colab_tri_resume.py`), cài đặt `pip install -e .` để kích hoạt bộ lệnh CLI:

```bash
# 1. Thu thập đa phiên bản & trích dẫn cho một dải arXiv ID thành viên (hỗ trợ --resume)
arxiv-harvester               --member-id 24120282               --start-month 2024-12 --start-id 18848               --end-month 2025-01 --end-id 2641               --raw-dir data/raw               --output-dir data/shards/24120282               --export-course --resume

# 2. Kiểm toán độc lập mã nguồn TeX/Bib/BBL & tải bù hỗ trợ HTTP Range
arxiv-audit-sources --publications data/shards --raw-dir data/raw --output data/audits/sources.parquet
arxiv-retry-sources --manifest data/audits/sources.retry.jsonl --output-dir data/repaired_sources --limit 20 --execute

# 3. Hợp nhất 5 phân vùng thành viên (25.000 bài) & xuất 2 bảng chuẩn M1 (papers, references)
python -m crawler.merge_shards --shards-dir data/shards --output-dir data/processed
```

## A.3. Tái lập phân tích cấu trúc & kiểm chuẩn Milestone 2 (Google Colab CPU-only)

1. **Thực thi theo từng thành viên trên Colab CPU Standard** (kiểm tra tự động không gắn `/dev/nvidia0` hay `COLAB_TPU_ADDR`): mở `colab_parser_full.ipynb` để điều phối `tools/m2_colab_runner.py` kèm `configs/parser_roots.json`, hoặc chạy trực tiếp qua CLI:

```bash
arxiv-parser               --publications data/processed/publications.parquet               --raw-dir data/raw               --output-dir data/parsed               --root-overrides configs/parser_roots.json               --resume

arxiv-validate-project               --course-dir data/course_export               --parsed-dir data/parsed               --publications data/processed/publications.parquet               --output data/parsed/project_validation.json               --require-full
```

2. **Hợp nhất & tổng hợp benchmark 25.000 bài báo**: chạy `tools/m2_team_consolidator.py` để gộp kết quả 5 thành viên, xác minh toàn vẹn ngoài bộ nhớ 5 bảng bằng SQLite (`structural_valid = True`), tổng hợp 5 phiên Colab CPU qua `src/crawler/benchmark_sessions.py` và trích xuất thống kê phục vụ Book bằng `tools/extract_book_aggregates_colab.py`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
import sys, importlib.metadata
print("Phiên bản Python biên dịch Book:", sys.version.split()[0])
pkgs = ["jupyter-book", "pandas", "pyarrow", "matplotlib", "nbformat", "arxiv-harvester"]
rows = []
for p in pkgs:
    try:
        ver = importlib.metadata.version(p)
    except importlib.metadata.PackageNotFoundError:
        ver = "editable/PYTHONPATH"
    rows.append({"Gói thư viện": p, "Phiên bản khi biên dịch": ver})
display(style_table(pd.DataFrame(rows), caption="Bảng A.1. Phiên bản các thư viện cốt lõi trong môi trường biên dịch Jupyter Book."))

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 1 — Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [Đặc tả chính thức Milestone 2 — Hierarchical Parsing & Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`pyproject.toml`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/pyproject.toml).
- Mã nguồn triển khai: [`colab_crawler.ipynb`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/colab_crawler.ipynb).
- Mã nguồn triển khai: [`colab_parser_full.ipynb`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/colab_parser_full.ipynb).
- Mã nguồn triển khai: [`configs/parser_roots.json`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/configs/parser_roots.json).
- Mã nguồn triển khai: [`tools/m1_colab_tri_resume.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m1_colab_tri_resume.py).
- Mã nguồn triển khai: [`tools/m2_colab_runner.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_colab_runner.py).
- Mã nguồn triển khai: [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).
- Mã nguồn triển khai: [`tools/extract_book_aggregates_colab.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/extract_book_aggregates_colab.py).